In [ ]:
from turtle import *
from datetime import datetime
import math

# ---------- Helpers (kept similar to your style) ----------
def jump(distance, angle=0):
    penup()
    right(angle)
    forward(distance)
    left(angle)
    pendown()

# ---------- Background (radial gradient using concentric filled circles) ----------
def draw_background(radius, layers=40):
    screen.tracer(False)
    bg = Turtle(visible=False)
    bg.hideturtle()
    bg.penup()
    colormode(255)

    # two colors to blend (outer -> inner)
    outer = (20, 35, 70)    # deep navy
    inner = (200, 230, 255) # pale sky

    for i in range(layers, 0, -1):
        t = i / layers
        r = int(outer[0] * t + inner[0] * (1 - t))
        g = int(outer[1] * t + inner[1] * (1 - t))
        b = int(outer[2] * t + inner[2] * (1 - t))
        bg.fillcolor(r, g, b)
        bg.goto(0, -radius * (i / layers))
        bg.pendown()
        bg.begin_fill()
        bg.circle(radius * (i / layers))
        bg.end_fill()
        bg.penup()

    screen.tracer(True)

# ---------- Clock face with ticks and numbers ----------
def clockface(radius):
    # draw face rim and inner face
    drawer = Turtle(visible=False)
    drawer.hideturtle()
    drawer.speed(0)
    drawer.penup()

    # shadow rim
    drawer.goto(0, -radius - 10)
    drawer.pendown()
    drawer.fillcolor("#111522")
    drawer.begin_fill()
    drawer.circle(radius + 10)
    drawer.end_fill()
    drawer.penup()

    # inner face
    drawer.goto(0, -radius)
    drawer.pendown()
    drawer.fillcolor("white")
    drawer.begin_fill()
    drawer.circle(radius)
    drawer.end_fill()
    drawer.penup()

    # ticks
    drawer.color("#222222")
    drawer.pensize(3)
    for i in range(60):
        angle = math.radians(i * 6)  # i*6 degrees
        # compute start and end points for ticks so we don't rely on reset/jump
        outer_r = radius - 8
        if i % 5 == 0:  # hour tick longer
            inner_r = radius - 40
            drawer.pensize(4)
        else:
            inner_r = radius - 20
            drawer.pensize(2)
        x1 = inner_r * math.cos(math.radians(90 - i * 6))
        y1 = inner_r * math.sin(math.radians(90 - i * 6))
        x2 = outer_r * math.cos(math.radians(90 - i * 6))
        y2 = outer_r * math.sin(math.radians(90 - i * 6))
        drawer.goto(x1, y1)
        drawer.pendown()
        drawer.goto(x2, y2)
        drawer.penup()

    # numbers 1..12
    drawer.color("#1a1a1a")
    drawer.penup()
    for n in range(1, 13):
        ang = math.radians((n % 12) * 30)  # 30 deg steps
        x = (radius - 70) * math.cos(math.radians(90 - (n % 12) * 30))
        y = (radius - 70) * math.sin(math.radians(90 - (n % 12) * 30))
        drawer.goto(x, y - 10)  # adjust vertical a bit
        drawer.write(str(n), align="center", font=("Helvetica", 18, "bold"))
    drawer.penup()

# ---------- Create arrow shape whose tail is at (0,0) so rotation is around tail ----------
def register_arrow_shape(name, length, shaft_width=14, head_len_ratio=0.22):
    # polygon coordinates: tail at (0,0); tip at (length, 0)
    head_len = length * head_len_ratio
    front = length - head_len
    w = shaft_width

    # Points in order (clockwise)
    pts = [
        (0, -w/2),
        (front, -w/2),
        (front, -w),
        (length, 0),
        (front, w),
        (front, w/2),
        (0, w/2)
    ]
    # Register shape; if same name exists, override by using a unique name or remove first.
    try:
        register_shape(name, pts)
    except Exception:
        # fallback: attempt to unregister by re-registering - turtle may raise if exists
        # We'll still ignore exceptions here; registration should succeed in common runtimes.
        pass

# ---------- Make hand turtle using the custom arrow shape ----------
def make_hand(name, length, color, shaft_width=14, head_ratio=0.22):
    register_arrow_shape(name, length, shaft_width, head_ratio)
    t = Turtle(visible=False)
    t.hideturtle()
    t.shape(name)
    t.shapesize(1, 1, 1)  # not scaling further; shape coords already in pixels
    t.color(color)
    t.penup()
    t.goto(0, 0)
    t.showturtle()
    t.speed(0)
    return t

# ---------- Update hands and the date/day text (clockwise motion) ----------
def update():
    now = datetime.now()
    sec = now.second + now.microsecond * 1e-6
    minute = now.minute + sec / 60.0
    hour = (now.hour % 12) + minute / 60.0

    # angles in degrees, clockwise from 12:00
    sec_angle = sec * 6.0
    min_angle = minute * 6.0
    hour_angle = hour * 30.0

    screen.tracer(False)

    # set heading so that 0 degrees means pointing to 12 o'clock and positive is clockwise:
    # turtle uses standard mode: 0 is east and positive angles are CCW.
    # mapping: desired_heading = 90 - clockwise_angle
    second_hand.setheading(90 - sec_angle)
    minute_hand.setheading(90 - min_angle)
    hour_hand.setheading(90 - hour_angle)

    # update central date/day
    writer.clear()
    writer.color("#6b0f0f")
    writer.goto(0, -18)
    writer.write(now.strftime("%A"), align="center", font=("Arial", 16, "bold"))
    writer.goto(0, -45)
    writer.write(now.strftime("%d %B %Y"), align="center", font=("Arial", 12, "normal"))

    screen.tracer(True)
    # smoother update (every 100 ms)
    ontimer(update, 100)

# ---------- Main configuration and creation ----------
screen = Screen()
screen.setup(800, 800)
screen.title("Beautiful Clock — clockwise, arrow hands")
screen.bgcolor("#0b1726")  # base while gradient draws; will be overwritten by draw_background
mode("standard")  # use standard coordinates: 0° = east, positive is CCW

RADIUS = 260

# draw gradient background then clock face on top

clockface(RADIUS)

# create hands (tail is at center because we made the polygon that way)
hour_hand   = make_hand("hour_arrow",   length=150, color="#222222", shaft_width=18, head_ratio=0.18)
minute_hand = make_hand("minute_arrow", length=210, color="#0b3d91", shaft_width=12, head_ratio=0.16)
second_hand = make_hand("second_arrow", length=240, color="#c62828", shaft_width=6,  head_ratio=0.28)

# center writer for date/day
writer = Turtle(visible=False)
writer.hideturtle()
writer.penup()
writer.goto(0, -30)

# small decorative center pin
pin = Turtle(visible=False)
pin.hideturtle()
pin.penup()
pin.goto(0, 0)
pin.dot(16, "#333333")
pin.dot(8, "#f4f4f4")

# start animation
update()
done()


Terminator: 